In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

In [0]:

df_transactions = spark.table('samples.bakehouse.sales_transactions')

In [0]:
df_transactions.display()

In [0]:
df_customers = spark.table('samples.bakehouse.sales_customers')

In [0]:
df_customers.display()

In [0]:
df_customers.filter("state = 'New York'").display()

# Spark's Query Plans

![Spark Execution](../images/spark-execution.png)


- https://bigdataperformance.substack.com/p/understanding-apache-sparks-execution
- https://medium.com/@Sanjay007/understanding-the-execution-process-of-apache-spark-4209374febb8
- https://www.analyticsvidhya.com/blog/2021/08/understand-the-internal-working-of-apache-spark/

# Narrow Transformations
- `filter` rows where `state='New York'`
- `add` a new column: adding `first_name` and `last_name`
- `select` relevant columns

In [0]:
df_narrow_transform = (
    df_customers
    .filter(F.col("state") == "New York")
    .withColumn("name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
    .withColumn("customerID", F.col('customerID').cast("Int") - 1000000)
)

df_narrow_transform.display()
df_narrow_transform.explain(True)

# Wide Transformations
1. Repartition
2. Coalesce
3. Joins
4. GroupBy
   - `count`
   - `countDistinct`
   - `sum`

## 1. Repartition

In [0]:
df_transactions.repartition(24).explain(True)

## 2. Coalesce

In [0]:
df_transactions.coalesce(5).explain(True)

## 3. Joins

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [0]:
df_narrow_transform.count()

In [0]:
df_transactions.count()

In [0]:
df_customers = (df_customers.withColumn("customerID", F.col('customerID').cast("Int") - 1000000))

In [0]:
df_joined = (
    df_transactions.join(
        df_customers,
        how="inner",
        on="customerID"
    )
)

In [0]:
df_joined.count()

In [0]:
df_joined.display()

In [0]:
df_joined.explain(True)

## 4. GroupBy

In [0]:
df_transactions.printSchema()

### GroupBy Count

In [0]:
df_state_counts = (
    df_joined
    .groupBy("state")
    .count()
)

In [0]:
df_state_counts.explain(True)

In [0]:
df_txn_amt_city = (
    df_joined
    .groupBy("state")
    .agg(F.sum("totalPrice").alias("txn_amt"))
)

In [0]:
df_txn_amt_city.explain(True)

### GroupBy Count Distinct 

In [0]:
df_txn_per_state = (
    df_joined
    .groupBy("customerID")
    .agg(F.countDistinct("state").alias("city_count"))
)

In [0]:
df_joined.display()

In [0]:
df_txn_per_state.display()
df_txn_per_state.explain(True)

# 5. Interesting Observations

In [0]:
df_customer_gt_50 = (
    df_customers
    .filter(F.col("age").cast("int") > 50)
)
df_customer_gt_50.show(5, False)
df_customer_gt_50.explain(True)